# Dart recognition pipeline

Stages: **1** load image, **2** detect the board, **3** rectify, **4** locate darts, **5** read numbers, **6** locate dart tips, **7** score, **8** combined view.

Stages 1-3 run the existing `dartcv` board detector. Stages 4-6 need recognition backends that are **not configured yet**; until one is plugged in, they report `not configured` and draw nothing. Nothing is invented: no boxes, no readings, no tips, and therefore no scores.

All results are in **original-image pixels** `(x, y)`. Only one image is processed at a time.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from dartcv.detect import DetectionError, detect_board
from dartcv.diagnostics import detection_overlays
from dartcv.geometry import SECTORS, compute_homography, project, score_from_image_point
from dartcv.overlay import draw_overlay, topdown_overlay


def show(img_bgr, title, width=7):
    plt.figure(figsize=(width, width * img_bgr.shape[0] / img_bgr.shape[1]))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## Configuration

In [ ]:
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())

IMAGE_PATH = REPO_ROOT / 'data/images/single_dart/PXL_20261018_152634348.jpg'
UPRIGHT_DEG = 0.0  # clockwise angle at which the 20 appears; resolves the 36-degree ambiguity
SHOW_DIAGNOSTICS = False  # also show every board-detection stage
MANUAL_CALIBRATION = None  # four (x, y) pixels for 20|1, 6|10, 3|19, 11|14 instead of auto-detection
MANUAL_TIPS = []  # optional hand-picked (x, y) tip pixels, scored with provenance 'manual'
EXPORT_PATH = None  # e.g. REPO_ROOT / 'out/recognition.png'; nothing is written when None

## 1. Load the image

In [ ]:
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f'cannot read {IMAGE_PATH}')
image.flags.writeable = False  # later stages must draw on copies

print(IMAGE_PATH.name, 'height x width x channels:', image.shape)
show(image, f'1. Original: {IMAGE_PATH.name}')

## 2. Detect the board

The homography `H` maps board millimetres to image pixels. Rings follow the measured paint edges where available. If the 20 is not near the top of the image, set `UPRIGHT_DEG`; if detection fails, set `MANUAL_CALIBRATION`.

In [ ]:
if MANUAL_CALIBRATION is not None:
    detection = None
    H = compute_homography(MANUAL_CALIBRATION)
    print('calibration: manual')
else:
    try:
        detection = detect_board(image, upright_deg=UPRIGHT_DEG, diagnostics=SHOW_DIAGNOSTICS)
    except DetectionError as exc:
        raise RuntimeError(f'board detection failed ({exc}); set MANUAL_CALIBRATION') from exc
    H = detection.H
    bull = 'found' if detection.bull is not None else 'not found'
    print(f'calibration: automatic (ECC score {detection.ecc_score}, bull {bull}, ring edges {len(detection.rings)}/4)')

ring_contours = {} if detection is None else {name: ring.contour for name, ring in detection.rings.items()}
board_overlay = draw_overlay(
    image, H, numbers=False, ring_contours=ring_contours, thickness=max(1, max(image.shape) // 1600)
)
show(board_overlay, '2. Board geometry')

if SHOW_DIAGNOSTICS and detection is not None:
    for name, view in detection_overlays(image, detection).items():
        show(view, name)

## 3. Rectify

In [ ]:
show(topdown_overlay(image, H, numbers=False, ring_contours=ring_contours), '3. Top-down view (20 at the top)')

## Result contracts and backend slots

Each recognition stage has a backend slot that stays `None` until a model is connected. A stage reports `not configured` (no backend), `failed` (backend raised or returned invalid data) or `ok`; `ok` with zero items is a genuine empty result.

- `DART_DETECTOR(image) -> list[DartBox]`
- `NUMBER_READER(image, H) -> list[NumberReading]`
- `TIP_LOCATOR(image, boxes) -> list[DartTip]`

Backends must return original-image pixels; undo any crop or resize before returning. A dart box is not a tip: only a `DartTip` is scored.

In [ ]:
@dataclass(frozen=True)
class DartBox:
    id: int
    xyxy: tuple[float, float, float, float]
    confidence: float
    provenance: str


@dataclass(frozen=True, eq=False)
class NumberReading:
    polygon: np.ndarray  # (N, 2) outline of the printed number
    text: str  # what was read, never the expected sector label
    confidence: float
    provenance: str


@dataclass(frozen=True)
class DartTip:
    dart_id: int | None  # None: not associated with a detected dart
    xy: tuple[float, float]
    confidence: float
    provenance: str


@dataclass(frozen=True)
class StageResult[T]:
    status: str  # 'ok', 'not configured' or 'failed'
    items: tuple[T, ...] = ()
    message: str = ''


DART_DETECTOR = None
NUMBER_READER = None
TIP_LOCATOR = None


def _inside(image, pts):
    pts = np.asarray(pts, float).reshape(-1, 2)
    h, w = image.shape[:2]
    return bool(np.isfinite(pts).all() and (pts >= 0).all() and (pts[:, 0] <= w - 1).all() and (pts[:, 1] <= h - 1).all())


def _check_confidence(item):
    if not 0.0 <= item.confidence <= 1.0:
        raise ValueError(f'confidence outside [0, 1]: {item}')


def validate_boxes(image, boxes):
    ids = [b.id for b in boxes]
    if len(set(ids)) != len(ids):
        raise ValueError('duplicate dart ids')
    for b in boxes:
        x0, y0, x1, y1 = b.xyxy
        if not (x0 < x1 and y0 < y1 and _inside(image, [(x0, y0), (x1, y1)])):
            raise ValueError(f'dart {b.id}: box is empty or outside the image')
        _check_confidence(b)


def validate_numbers(image, readings):
    for r in readings:
        polygon = np.asarray(r.polygon, float)
        if polygon.ndim != 2 or polygon.shape[1] != 2 or len(polygon) < 3 or not _inside(image, polygon):
            raise ValueError(f'number {r.text!r}: polygon is not a valid outline inside the image')
        if not r.text.strip():
            raise ValueError('empty number text')
        _check_confidence(r)


def validate_tips(image, tips, boxes):
    boxes_by_id = {b.id: b for b in boxes}
    for t in tips:
        if not _inside(image, [t.xy]):
            raise ValueError(f'tip {t.xy} is outside the image')
        _check_confidence(t)
        if t.dart_id is None:
            continue
        if t.dart_id not in boxes_by_id:
            raise ValueError(f'tip refers to unknown dart {t.dart_id}')
        x0, y0, x1, y1 = boxes_by_id[t.dart_id].xyxy
        pad = 0.1 * max(x1 - x0, y1 - y0)  # tips sit at the box edge
        if not (x0 - pad <= t.xy[0] <= x1 + pad and y0 - pad <= t.xy[1] <= y1 + pad):
            raise ValueError(f'tip {t.xy} is far from the box of dart {t.dart_id}')


def run_stage(name, backend, validate, *args):
    if backend is None:
        return StageResult('not configured', message=f'{name}: no backend configured')
    try:
        items = tuple(backend(*args))
        validate(items)
    except Exception as exc:
        return StageResult('failed', message=f'{name}: {type(exc).__name__}: {exc}')
    return StageResult('ok', items, f'{name}: {len(items)} found')


def expected_sector(H, xy):
    x, y = project(np.linalg.inv(H), [xy])[0]
    return SECTORS[int(((np.degrees(np.arctan2(x, -y)) + 9) % 360) // 18) % 20]


def annotate(base, boxes=(), numbers=(), tips=(), scores=None):
    out = base.copy()
    k = max(1, round(max(out.shape[:2]) / 1600))

    def label(text, xy, colour):
        pos = (int(xy[0]), int(xy[1]))
        cv2.putText(out, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7 * k, (0, 0, 0), 4 * k, cv2.LINE_AA)
        cv2.putText(out, text, pos, cv2.FONT_HERSHEY_SIMPLEX, 0.7 * k, colour, 2 * k, cv2.LINE_AA)

    for b in boxes:
        x0, y0, x1, y1 = (int(round(v)) for v in b.xyxy)
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 200, 255), 2 * k)
        label(f'dart {b.id} {b.confidence:.2f}', (x0, max(y0 - 8 * k, 0)), (0, 200, 255))
    for n in numbers:
        polygon = np.asarray(n.polygon, float)
        cv2.polylines(out, [np.round(polygon).astype(np.int32)], True, (255, 200, 0), 2 * k, cv2.LINE_AA)
        label(n.text, polygon.min(axis=0), (255, 200, 0))
    for i, t in enumerate(tips):
        x, y = int(round(t.xy[0])), int(round(t.xy[1]))
        cv2.drawMarker(out, (x, y), (0, 0, 255), cv2.MARKER_CROSS, 30 * k, 3 * k, cv2.LINE_AA)
        if scores is not None:
            label(str(scores[i]), (x + 12 * k, y - 12 * k), (0, 0, 255))
    return out


def show_stage(title, result, base, **annotations):
    print(f'{title}: {result.message}')
    if result.status == 'ok':
        show(annotate(base, **annotations), title)

## 4. Locate darts

In [ ]:
darts = run_stage('dart detection', DART_DETECTOR, lambda items: validate_boxes(image, items), image)
show_stage('4. Dart locations', darts, image, boxes=darts.items)

## 5. Read the numbers

`NUMBER_READER` must return a box (`polygon`) and the recognised `text` for each printed number; the boxes are drawn on the full image, not cropped out. No number detector is configured yet, so this stage reports `not configured`. Readings are only compared with the model's sector order; a mismatch is reported, never overwritten. A systematic mismatch usually means the board orientation is wrong (see `UPRIGHT_DEG`).

In [ ]:
numbers = run_stage('number reading', NUMBER_READER, lambda items: validate_numbers(image, items), image, H)
show_stage('5. Number readings', numbers, image, numbers=numbers.items)

for r in numbers.items:
    expected = expected_sector(H, np.asarray(r.polygon, float).mean(axis=0))
    verdict = 'matches' if r.text == str(expected) else 'DIFFERS from'
    print(f'read {r.text!r} ({r.confidence:.2f}) {verdict} the model sector {expected}')

## 6. Locate dart tips

Tips come from `TIP_LOCATOR` or from `MANUAL_TIPS`; a box centre is never used as a tip.

In [ ]:
tip_stage = run_stage('tip localisation', TIP_LOCATOR, lambda items: validate_tips(image, items, darts.items), image, darts.items)
print(tip_stage.message)

manual_tips = tuple(DartTip(None, (float(x), float(y)), 1.0, 'manual') for x, y in MANUAL_TIPS)
validate_tips(image, manual_tips, ())
tips = tip_stage.items + manual_tips
print(f'{len(tips)} tip(s): {len(tip_stage.items)} detected, {len(manual_tips)} manual')

if tips:
    show(annotate(image, boxes=darts.items, tips=tips), '6. Dart tips')

## 7. Score

Scores use the standard board radii, not the fitted paint contours, and inherit the calibration's uncertainty. A score close to a ring or wire boundary is not reliable.

In [ ]:
scores = [score_from_image_point(H, t.xy) for t in tips]
if tips:
    print(f'{"dart":>5} {"tip (x, y)":>20} {"conf":>5} {"source":>8} {"score":>5}')
    for t, s in zip(tips, scores):
        dart = '-' if t.dart_id is None else t.dart_id
        print(f'{dart:>5} {t.xy[0]:>9.1f}, {t.xy[1]:>8.1f} {t.confidence:>5.2f} {t.provenance:>8} {s:>5}')
    print('total:', sum(scores))
else:
    print('No dart tips available: scores are unknown.')

## 8. Combined view

In [ ]:
combined = annotate(
    board_overlay, boxes=darts.items, numbers=numbers.items, tips=tips, scores=scores if tips else None
)
show(combined, '8. Board, darts, numbers, tips and scores')

if EXPORT_PATH is not None:
    EXPORT_PATH = Path(EXPORT_PATH)
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(EXPORT_PATH), combined):
        raise OSError(f'could not write {EXPORT_PATH}')
    print('wrote', EXPORT_PATH)

## Connecting recognition models

Assign a callable to `DART_DETECTOR`, `NUMBER_READER` or `TIP_LOCATOR` in the contracts cell and re-run. Open choices: a dart detector (boxes), a printed-number OCR, and, for real scoring, a model that localises the dart **tip**, since a box does not determine the scoring wedge. Resolving the 36-degree board orientation from reliable OCR is a later step.